# From Actor-Critic to PPO — Hands-on Exercise

In this notebook you will turn a **simple Actor-Critic (A2C)** agent into a **Proximal Policy Optimization (PPO)** agent, step by step, on the classic **Pendulum** control task (continuous actions).

### How this notebook works
1. **Part 1 (provided):** a minimal A2C baseline. You just run it and observe that it learns *slowly and unstably*.
2. **Part 2 (your turn):** you implement the pieces that make PPO work:
   - **TODO 1** — the agent ↔ environment **interaction** (`collect_rollout`)
   - **TODO 2** — **Generalized Advantage Estimation (GAE)**
   - **TODO 3** — the **clipped surrogate** (the PPO actor loss)
   - **TODO 4** — the separate **actor** and **critic** update functions
   - **TODO 5** — the **multi-epoch, mini-batch** training loop

Each task has a markdown block above the code with **what to do**, the **inputs** (name / shape / meaning), the **expected output**, and — when there is any — the **math**.

> The math lives in the **markdown**. In the code you only get short hints about *which variable to assign*, never the formula itself. Translate the math yourself.

Most tasks are followed by a ✅ **sanity-check cell** — run it to confirm your work before moving on. Look for the 🟦 **TODO** markers.

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions.normal import Normal
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# Simple in-notebook renderer (provided — no need to modify).
def make_renderer(env):
    """Display the env and return a `show()` function to refresh it in place."""
    fig, ax = plt.subplots()
    ax.axis("off")
    img = ax.imshow(env.render())
    handle = display(fig, display_id=True)
    plt.close(fig)  # avoid a duplicate static figure

    def show():
        img.set_data(env.render())
        handle.update(fig)

    return show

## The environment: Pendulum-v1

- **Goal:** swing the pendulum up and keep it balanced upright.
- **Observation** (shape `(3,)`): `[cos(theta), sin(theta), angular_velocity]`.
- **Action** (shape `(1,)`): a torque in `[-2, 2]`.
- **Reward:** close to `0` when upright with little effort, very negative otherwise. An episode lasts 200 steps, so the best return is near `0`, while a poor policy gets roughly `-1200` to `-1600`.

In [ ]:
env = gym.make("Pendulum-v1", render_mode="rgb_array")
state, info = env.reset()
make_renderer(env)  # show the current frame
print("observation space:", env.observation_space)
print("action space:", env.action_space)

## Part 1 — A simple Actor-Critic baseline (provided)

An **Actor-Critic** agent has two networks that learn together:

```text
            +---------------+   action a ~ pi(a|s)    +-----------+
   state s  |     ACTOR     | ----------------------> |           |
   +------> |    pi(a|s)    |                         |    ENV    |
   |        +---------------+                         |           |
   |                                                  +-----+-----+
   |                          reward r , next state s'      |
   | <-----------------------------------------------------+
   |        +---------------+
   +------> |    CRITIC     | -->  V(s)
            |     V(s)      |
            +---------------+

   advantage:  A(s,a) = r + gamma * V(s') - V(s)

   * the CRITIC learns to predict V(s)  ->  how good a state is
   * the ACTOR makes actions with POSITIVE advantage more likely,
     and actions with NEGATIVE advantage less likely
```

This baseline is *bare-bones*:
- a **single** environment,
- it collects a few episodes, does **one** gradient step on that data, then throws it away,
- the advantage is the simplest possible **one-step TD error** $A = r + \gamma V(s') - V(s)$,
- **no** GAE, **no** clipping, **no** multiple epochs / mini-batches.

Run the cells below — **you don't need to change anything here**. Just notice at the end how noisy and unreliable the learning is. That is exactly the problem PPO will fix.

In [ ]:
class ActorNetwork(nn.Module):
    """Outputs a Normal distribution over actions (mean in (-1,1), positive std)."""
    def __init__(self, obs_size, num_actions):
        super().__init__()
        self.layer1 = nn.Linear(obs_size, 64)
        self.layer2 = nn.Linear(64, 64)
        self.mu = nn.Linear(64, num_actions)
        self.std = nn.Linear(64, num_actions)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        mu = torch.tanh(self.mu(x))            # mean in (-1, 1)
        std = F.softplus(self.std(x)) + 1e-4   # std > 0
        return Normal(mu, std)


class CriticNetwork(nn.Module):
    """Estimates the value V(s) of a state."""
    def __init__(self, obs_size):
        super().__init__()
        self.layer1 = nn.Linear(obs_size, 64)
        self.layer2 = nn.Linear(64, 64)
        self.v = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        return self.v(x)

In [ ]:
# Provided helpers (used by the baseline and later by your rollout).
def get_action(actor, state):
    """Sample an action and its log-prob. Works on a single state or a batch."""
    pi = actor(state)
    action = pi.sample()
    action = torch.clamp(action, -1.0, 1.0)
    log_prob = pi.log_prob(action).sum(dim=-1)
    return action, log_prob

def scale_action(action, env):
    """Map an action from (-1, 1) to the env's real range (-2, 2 for Pendulum)."""
    high = env.action_space.high.max()
    low = env.action_space.low.min()
    return np.clip(action.numpy() * high, low, high)

In [ ]:
def simple_a2c(env, num_iterations=500, episodes_per_iter=3, gamma=0.9,
               lr_actor=5e-5, lr_critic=5e-4):
    """Minimal Actor-Critic: single env, one-step TD advantage, ONE update per batch."""
    obs_size = env.observation_space.shape[0]
    num_actions = env.action_space.shape[0]
    actor = ActorNetwork(obs_size, num_actions)
    critic = CriticNetwork(obs_size)
    actor_opt = torch.optim.Adam(actor.parameters(), lr=lr_actor)
    critic_opt = torch.optim.Adam(critic.parameters(), lr=lr_critic)

    episode_returns = []
    for iteration in range(1, num_iterations + 1):
        states, actions, rewards, next_states, dones = [], [], [], [], []

        # ---- collect a few full episodes with the current policy ----
        for _ in range(episodes_per_iter):
            state, _ = env.reset()
            done = False
            ep_return = 0.0
            while not done:
                state_t = torch.tensor(state, dtype=torch.float32)
                with torch.no_grad():
                    action, _ = get_action(actor, state_t)
                next_state, reward, term, trunc, _ = env.step(scale_action(action, env))
                done = term or trunc
                states.append(state_t)
                actions.append(action)
                rewards.append(float(reward))
                next_states.append(torch.tensor(next_state, dtype=torch.float32))
                dones.append(float(done))
                state = next_state
                ep_return += reward
            episode_returns.append(ep_return)

        # ---- stack to tensors ----
        states = torch.stack(states)
        actions = torch.stack(actions)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        next_states = torch.stack(next_states)
        dones = torch.tensor(dones, dtype=torch.float32)

        # ---- one-step TD advantage: A = r + gamma * V(s') - V(s) ----
        values = critic(states).squeeze(-1)
        next_values = critic(next_states).squeeze(-1)
        td_target = rewards + gamma * (1 - dones) * next_values.detach()
        advantages = (td_target - values).detach()

        # ---- ONE gradient step (no epochs, no minibatch, no clipping) ----
        pi = actor(states)
        log_probs = pi.log_prob(actions).sum(dim=-1)
        actor_loss = -(log_probs * advantages).mean()
        actor_opt.zero_grad(); actor_loss.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), 0.5); actor_opt.step()

        critic_loss = F.mse_loss(values, td_target)
        critic_opt.zero_grad(); critic_loss.backward()
        nn.utils.clip_grad_norm_(critic.parameters(), 1.0); critic_opt.step()

        if iteration % 50 == 0:
            print(f"iter {iteration}  avg return (last 50 eps): {np.mean(episode_returns[-50:]):.1f}")

    return actor, critic, episode_returns

In [ ]:
def running_mean(x, N=50):
    cumsum = np.cumsum(np.insert(x, 0, 0))
    return (cumsum[N:] - cumsum[:-N]) / float(N)

a2c_actor, a2c_critic, a2c_returns = simple_a2c(env, num_iterations=500)

plt.plot(running_mean(a2c_returns))
plt.xlabel("episode"); plt.ylabel("return (running mean)")
plt.title("Simple A2C baseline")
plt.show()

**What you should see:** the curve goes up *somewhat*, but it is **noisy and unstable** — it often stalls, dips, or fails to reach a good policy, and the outcome changes a lot between runs.

Why? Each batch of data is used for a **single** gradient step and then discarded (sample-inefficient), the **one-step** advantage is high-variance, and nothing prevents a single update from moving the policy too far and destroying it.

PPO fixes all three. Let's build it. 👇

## Part 2 — Build PPO (your turn)

PPO keeps the same actor & critic, but changes **how** they are updated:

```text
  +----------------------------------------------------------------+
  | 1. COLLECT a rollout with the current ("old") policy pi_old    |
  |    store per step: state, action, reward, done, V(s), logp_old |
  +-------------------------------+--------------------------------+
                                  |
                                  v
  +----------------------------------------------------------------+
  | 2. GAE: combine rewards and V(s) into low-variance advantages A|
  +-------------------------------+--------------------------------+
                                  |
                                  v
  +----------------------------------------------------------------+
  | 3. UPDATE for several EPOCHS over MINI-BATCHES of the SAME data |
  |      ratio   r = pi_new(a|s) / pi_old(a|s)                     |
  |      ACTOR   maximize  min( r*A , clip(r, 1-eps, 1+eps)*A )    |
  |      CRITIC  minimize  ( V(s) - return )^2                     |
  |                                                                |
  |    clipping stops pi_new from moving too far from pi_old,      |
  |    so we can safely REUSE each rollout many times -> stable!   |
  +----------------------------------------------------------------+
```

The three ideas, mapped to your TODOs:
- **GAE** (TODO 2) → a lower-variance advantage estimate.
- **Clipped surrogate** (TODO 3) → reuse data without the policy moving too far.
- **Multiple epochs over mini-batches** (TODO 4 & 5) → many stable updates per rollout.

We also switch to **vectorized environments** (many envs in parallel), and you implement the **interaction** that collects the data (TODO 1).

In [ ]:
NUM_ENVS = 16
envs = gym.make_vec("Pendulum-v1", num_envs=NUM_ENVS, max_episode_steps=200)
print("single observation space:", envs.single_observation_space)
print("single action space:", envs.single_action_space)

### 🟦 TODO 1 — The interaction: collect a rollout

Implement the loop that makes the agent **interact** with the (vectorized) environment for `num_steps` and stores everything PPO needs.

**At each step you must:**
- ask the **actor** for an action and its log-prob,
- ask the **critic** for the value `V(s)`,
- **step all the parallel envs** with that action,
- store the transition.

Since we are only *acting* (not training) here, do it inside `torch.no_grad()`.

**Inputs**
- `envs`: the vectorized environments
- `actor`, `critic`: the two networks
- `state`: the current batch of observations, shape `(num_envs, obs_size)`
- `num_steps`: how many steps to collect

**Expected output** — a `rollout` dict of stacked tensors, plus the last `state`:
- `states`    `(num_steps, num_envs, obs_size)`
- `actions`   `(num_steps, num_envs, num_actions)`
- `log_probs` `(num_steps, num_envs)` — log-prob under the policy that collected it ("old" policy)
- `rewards`   `(num_steps, num_envs)`
- `dones`     `(num_steps, num_envs)`
- `values`    `(num_steps, num_envs)` — the critic's `V(s)` at collection time

Helpers you can use: `get_action(actor, state_t)`, `critic(state_t)`, `scale_action(action, envs)`. The stacking into the dict is already done for you.

In [ ]:
def collect_rollout(envs, actor, critic, state, num_steps):
    states, actions, log_probs, rewards, dones, values = [], [], [], [], [], []

    for _ in range(num_steps):
        state_t = torch.tensor(state, dtype=torch.float32)

        # ===== YOUR CODE HERE (TODO 1) =====
        # Wrap the network calls in `with torch.no_grad():`
        #   - get `action` and `log_prob` from the actor
        #   - get `value` from the critic (remember to .squeeze(-1))
        # Then step the environment:
        #   - `next_state, reward, term, trunc, _ = envs.step(scale_action(action, envs))`
        #   - combine `term` and `trunc` into a single `done`
        # Finally append to the six lists below. Store `reward` and `done` as
        # float tensors, e.g. `torch.tensor(reward, dtype=torch.float32)`, and
        # update `state = next_state`.
        ...
        # ===================================

    rollout = {
        "states": torch.stack(states),
        "actions": torch.stack(actions),
        "log_probs": torch.stack(log_probs),
        "rewards": torch.stack(rewards),
        "dones": torch.stack(dones),
        "values": torch.stack(values),
    }
    return rollout, state

In [ ]:
# ✅ Sanity check for TODO 1
_test_envs = gym.make_vec("Pendulum-v1", num_envs=2)
_a, _c = ActorNetwork(3, 1), CriticNetwork(3)
_s, _ = _test_envs.reset()
_roll, _s_last = collect_rollout(_test_envs, _a, _c, _s, num_steps=5)

_expected = {"states": (5, 2, 3), "actions": (5, 2, 1), "log_probs": (5, 2),
             "rewards": (5, 2), "dones": (5, 2), "values": (5, 2)}
for k, shape in _expected.items():
    assert _roll[k].shape == torch.Size(shape), f"{k}: got {tuple(_roll[k].shape)}, expected {shape}"
assert _s_last.shape == (2, 3)
_test_envs.close()
print("✅ TODO 1 looks correct! shapes:", {k: tuple(v.shape) for k, v in _roll.items()})

### 🟦 TODO 2 — Generalized Advantage Estimation (GAE)

GAE is a low-variance estimate of the advantage, computed by walking **backwards** through the rollout.

**Math.** For each timestep `t` (from last to first):

1. TD error: $\;\delta_t = r_t + \gamma\, V(s_{t+1})\,(1 - d_t) - V(s_t)$
2. advantage (recursive): $\;A_t = \delta_t + \gamma\, \lambda\, (1 - d_t)\, A_{t+1}$

where $d_t$ is the done flag (so we don't bootstrap across episode boundaries) and $A_{t+1}=0$ past the end of the rollout. The critic target is then $R_t = A_t + V(s_t)$.

**Inputs**
- `rewards`, `values`, `dones`: tensors `(num_steps, num_envs)`
- `last_value`: tensor `(num_envs,)` — $V(s_{t+1})$ for the very last step
- `gamma`, `lam`: scalars ($\gamma$, $\lambda$)

**Expected output**
- `advantages`, `returns`: tensors `(num_steps, num_envs)`

The variables `next_value` (= $V(s_{t+1})$) and `not_done` (= $1 - d_t$) are prepared for you. Fill the three marked lines.

In [ ]:
def compute_gae(rewards, values, dones, last_value, gamma=0.9, lam=0.95):
    num_steps = rewards.shape[0]
    advantages = torch.zeros_like(rewards)
    last_advantage = torch.zeros_like(last_value)   # A_{t+1}, starts at 0

    for t in reversed(range(num_steps)):
        next_value = last_value if t == num_steps - 1 else values[t + 1]
        not_done = 1.0 - dones[t]

        # ===== YOUR CODE HERE (TODO 2) =====
        delta = None             # the TD error for step t
        last_advantage = None    # the GAE advantage for step t
        advantages[t] = last_advantage
        # ===================================

    returns = advantages + values
    return advantages, returns

In [ ]:
# ✅ Sanity check for TODO 2
# With gamma=1 and lam=1, the advantage reduces to (sum of future rewards + last_value - V(s_t)).
_rewards = torch.tensor([[1.0], [2.0], [3.0]])   # (3 steps, 1 env)
_values  = torch.tensor([[0.5], [0.5], [0.5]])
_dones   = torch.zeros(3, 1)
_last_value = torch.tensor([0.0])

_adv, _ret = compute_gae(_rewards, _values, _dones, _last_value, gamma=1.0, lam=1.0)
print("advantages:", _adv.squeeze().tolist(), " (expected [5.5, 4.5, 2.5])")
print("returns:   ", _ret.squeeze().tolist(), " (expected [6.0, 5.0, 3.0])")
assert torch.allclose(_adv.squeeze(), torch.tensor([5.5, 4.5, 2.5]))
assert torch.allclose(_ret.squeeze(), torch.tensor([6.0, 5.0, 3.0]))
print("✅ TODO 2 looks correct!")

### 🟦 TODO 3 — The clipped surrogate (PPO actor loss)

PPO updates the policy using the **probability ratio** between the new and the old policy:

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)} = \exp\big(\log\pi_\theta(a_t|s_t) - \log\pi_{\theta_{old}}(a_t|s_t)\big)$$

The **clipped surrogate objective** to *maximize* is:

$$L^{CLIP} = \mathbb{E}_t\big[\min\big(r_t\,A_t,\ \ \mathrm{clip}(r_t,\,1-\epsilon,\,1+\epsilon)\,A_t\big)\big]$$

Because optimizers **minimize**, return the **negative**: $\;\text{loss} = -\,\text{mean}\big(\min(\text{surr1},\,\text{surr2})\big)$, where `surr1` is the unclipped term $r_t A_t$ and `surr2` is the clipped term.

**Inputs** (all shape `(batch,)` except the scalar)
- `new_log_probs` — $\log\pi_\theta(a|s)$ under the *current* policy
- `old_log_probs` — $\log\pi_{\theta_{old}}(a|s)$ stored during collection
- `advantages` — from GAE (already normalized)
- `clip_eps` — scalar $\epsilon$ (e.g. `0.2`)

**Expected output:** a single **scalar** tensor (the loss to minimize).

Useful: `torch.exp`, `torch.clamp(x, low, high)`, `torch.min(a, b)`, `.mean()`.

In [ ]:
def ppo_actor_loss(new_log_probs, old_log_probs, advantages, clip_eps=0.2):
    # ===== YOUR CODE HERE (TODO 3) =====
    ratio = None    # probability ratio new/old
    surr1 = None    # unclipped term
    surr2 = None    # clipped term
    loss = None     # negative mean of the elementwise minimum
    # ===================================
    return loss

In [ ]:
# ✅ Sanity check for TODO 3
# (a) unchanged policy -> ratio = 1, so loss = -mean(advantages)
_loss = ppo_actor_loss(torch.zeros(4), torch.zeros(4),
                       torch.tensor([1.0, -1.0, 2.0, 0.5]), clip_eps=0.2)
print("loss with ratio=1:", float(_loss), " (expected -0.625)")
assert torch.allclose(_loss, torch.tensor(-0.625))

# (b) a good action made much more likely -> ratio clipped to 1+eps = 1.2, so loss = -1.2
_loss = ppo_actor_loss(torch.tensor([1.0]), torch.zeros(1), torch.tensor([1.0]), clip_eps=0.2)
print("clipped loss:    ", float(_loss), " (expected -1.2)")
assert torch.allclose(_loss, torch.tensor(-1.2))
print("✅ TODO 3 looks correct!")

### 🟦 TODO 4 — Separate actor and critic updates

Write two small functions, each performing **one gradient step** on a single mini-batch. Keeping the actor and critic updates separate makes the training loop clean and easy to read.

**`update_actor(actor, actor_opt, states, actions, old_log_probs, advantages, clip_eps)`**
- run the actor on `states`, get the log-prob of `actions` under the current policy (sum over the action dimension) → `new_log_probs`,
- compute the loss with your `ppo_actor_loss` from TODO 3,
- take one gradient step (exact optimizer calls are given in the comments).

**`update_critic(critic, critic_opt, states, returns)`**
- run the critic on `states` to predict the values (shape `(batch,)`),
- loss = **MSE** between predicted values and `returns` (`F.mse_loss`),
- take one gradient step.

All inputs are flat mini-batch tensors. Each function returns its scalar loss value (handy for logging).

In [ ]:
def update_actor(actor, actor_opt, states, actions, old_log_probs, advantages, clip_eps):
    # ===== YOUR CODE HERE (TODO 4a) =====
    # 1) new_log_probs = log-prob of `actions` under actor(states), summed over the action dim
    # 2) actor_loss = ppo_actor_loss(new_log_probs, old_log_probs, advantages, clip_eps)
    # 3) take one gradient step:
    #        actor_opt.zero_grad()
    #        actor_loss.backward()
    #        nn.utils.clip_grad_norm_(actor.parameters(), 0.5)
    #        actor_opt.step()
    ...
    # ===================================
    return actor_loss.item()


def update_critic(critic, critic_opt, states, returns):
    # ===== YOUR CODE HERE (TODO 4b) =====
    # 1) values = critic(states).squeeze(-1)
    # 2) critic_loss = F.mse_loss(values, returns)
    # 3) take one gradient step:
    #        critic_opt.zero_grad()
    #        critic_loss.backward()
    #        nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
    #        critic_opt.step()
    ...
    # ===================================
    return critic_loss.item()

### 🟦 TODO 5 — The multi-epoch, mini-batch training loop

This is what makes PPO **sample-efficient and stable**: reuse each rollout for several **epochs**, and within each epoch update on shuffled **mini-batches**.

The data is already **flattened** and the advantages **normalized** for you. Your job is to write the **two for-loops**:
- repeat for `epochs` passes over the data,
- in each pass, shuffle the indices with `torch.randperm(batch_size)` and split them into mini-batches of size `minibatch_size`,
- for each mini-batch of indices `mb`, call:
  - `update_actor(actor, actor_opt, states[mb], actions[mb], old_log_probs[mb], advantages[mb], clip_eps)`
  - `update_critic(critic, critic_opt, states[mb], returns[mb])`

No new math here — just the loop structure. (Keep the last losses around so the function can return them for logging.)

In [ ]:
def ppo_update(actor, critic, actor_opt, critic_opt, rollout, advantages, returns,
               clip_eps=0.2, epochs=10, num_minibatches=4):
    # --- provided: flatten (num_steps, num_envs, ...) -> (batch, ...) ---
    obs_size = rollout["states"].shape[-1]
    act_dim = rollout["actions"].shape[-1]
    states = rollout["states"].reshape(-1, obs_size)
    actions = rollout["actions"].reshape(-1, act_dim)
    old_log_probs = rollout["log_probs"].reshape(-1)
    advantages = advantages.reshape(-1)
    returns = returns.reshape(-1)

    # normalize advantages (helps stability)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    batch_size = states.shape[0]
    minibatch_size = batch_size // num_minibatches

    actor_loss = critic_loss = None
    # ===== YOUR CODE HERE (TODO 5) =====
    # Write the epoch loop and, inside it, the mini-batch loop.
    # For each mini-batch of indices `mb`, call update_actor(...) and update_critic(...)
    # with the mini-batch slices, and store their returned loss values.
    ...
    # ===================================

    return actor_loss, critic_loss

### The PPO orchestrator (provided)

This ties everything together: **collect → GAE → update**, repeated for several iterations. It calls *your* `collect_rollout`, `compute_gae` and `ppo_update`, so it only works once TODOs 1–5 are done. You don't need to edit it.

In [ ]:
def ppo(envs, num_iterations=300, num_steps=256, gamma=0.9, lam=0.95,
        clip_eps=0.2, epochs=10, num_minibatches=4, lr_actor=3e-4, lr_critic=1e-3):
    obs_size = envs.single_observation_space.shape[0]
    num_actions = envs.single_action_space.shape[0]
    num_envs = envs.num_envs

    actor = ActorNetwork(obs_size, num_actions)
    critic = CriticNetwork(obs_size)
    actor_opt = torch.optim.Adam(actor.parameters(), lr=lr_actor)
    critic_opt = torch.optim.Adam(critic.parameters(), lr=lr_critic)

    state, _ = envs.reset()
    episode_returns = []
    running_returns = np.zeros(num_envs)

    for iteration in range(1, num_iterations + 1):
        rollout, state = collect_rollout(envs, actor, critic, state, num_steps)

        with torch.no_grad():
            last_value = critic(torch.tensor(state, dtype=torch.float32)).squeeze(-1)
        advantages, returns = compute_gae(rollout["rewards"], rollout["values"],
                                          rollout["dones"], last_value, gamma, lam)

        actor_loss, critic_loss = ppo_update(actor, critic, actor_opt, critic_opt,
                                             rollout, advantages, returns,
                                             clip_eps, epochs, num_minibatches)

        # logging: record the return of every finished episode
        for t in range(num_steps):
            running_returns += rollout["rewards"][t].numpy()
            for i in np.where(rollout["dones"][t].numpy().astype(bool))[0]:
                episode_returns.append(running_returns[i])
                running_returns[i] = 0.0

        if iteration % 10 == 0:
            avg = np.mean(episode_returns[-100:]) if episode_returns else float("nan")
            print(f"iter {iteration}  avg return: {avg:.1f}")

    return actor, critic, episode_returns

In [ ]:
ppo_actor, ppo_critic, ppo_returns = ppo(envs, num_iterations=300)

plt.plot(running_mean(ppo_returns))
plt.xlabel("episode"); plt.ylabel("return (running mean)")
plt.title("PPO")
plt.show()

**Compare** this curve with the A2C baseline from Part 1: PPO should climb faster, reach a clearly better return (close to `0`, i.e. an upright pendulum), and be far more stable across runs.

In [ ]:
# Watch the trained PPO policy
eval_env = gym.make("Pendulum-v1", render_mode="rgb_array")
state, info = eval_env.reset()
show = make_renderer(eval_env)

done = False
total = 0.0
while not done:
    with torch.no_grad():
        pi = ppo_actor(torch.tensor(state, dtype=torch.float32))
        action = torch.clamp(pi.mean, -1.0, 1.0) * eval_env.action_space.high.max()
    state, reward, term, trunc, _ = eval_env.step(action.numpy())
    done = term or trunc
    show()
    total += reward

eval_env.close()
print("evaluation return:", total)

### 🎯 Bonus / things to try
- Add an **entropy bonus** to the actor loss (`+ entropy_coef * pi.entropy()`) to encourage exploration.
- Sweep `clip_eps`, `epochs`, `num_minibatches`, `num_steps` and observe the effect on stability.
- Try different `lam` values for GAE (e.g. `0.9` vs `1.0`) and observe the bias/variance trade-off.
- Compare sample efficiency (return vs. number of environment steps) against the A2C baseline.